In [ ]:
import tifffile as tiff
import os
import pickle
from sklearn.decomposition import NMF 

In [2]:
from nmf_utils import compute_dff, expand_to_full_maps

In [3]:
def main(file_data,file_mask,path_save,file_stem, n_components=100):
    """
    Perform NMF-based decomposition on a masked TIFF movie and save results.

    This function:
    1. Loads a TIFF movie and corresponding binary mask.
    2. Computes ΔF/F (dF/F) signals within the masked region.
    3. Prepares the data for Non-negative Matrix Factorization (NMF).
    4. Runs NMF to extract temporal (S) and spatial (A) components.
    5. Reconstructs spatial maps into full image space.
    6. Stores results and metadata in a serialized bundle (.pkl).

    Parameters
    ----------
    file_data : str
        Path to the input TIFF movie file (time x height x width).
    file_mask : str
        Path to the mask image file (same spatial dimensions as movie).
    path_save : str
        Directory where the output file will be saved.
    file_stem : str
        Base filename used to construct the output filename.
    n_components : int, optional (default=100)
        Number of NMF components to extract.

    Returns
    -------
    None
        Results are saved to disk as a pickle file:
        '{file_stem}_NMF_no_filt.pkl'.

    Outputs (saved in bundle)
    -------------------------
    nmf : sklearn.decomposition.NMF
        Fitted NMF model.
    A : np.ndarray
        Spatial components (P x K), where P is number of masked pixels.
    S : np.ndarray
        Temporal components (T x K), where T is number of frames.
    spatial_maps : np.ndarray
        Spatial maps reconstructed into full image dimensions.
    mask : np.ndarray
        Binary mask used for analysis.
    mean_image : np.ndarray
        Mean projection of the movie.
    meta : dict
        Metadata including preprocessing parameters and data info.

    Notes
    -----
    - Input data is shifted to ensure non-negativity before NMF.
    - NMF uses coordinate descent solver with Frobenius norm loss.
    - dF/F is computed using a percentile baseline (default 10th percentile).
    """

    #load movie and mask
    movie = tiff.memmap(file_data)
    mask=tiff.imread(file_mask)
    mask = mask > 0
    mean_img = movie.mean(axis=0)

    #compute dF/F
    dff_masked, _ = compute_dff(movie, mask, percentile=10, return_full=False)

    # X for NMF: shape (T, P)
    X = dff_masked
    shift = X.min(axis=0, keepdims=True)   # per-pixel minimum over time
    X_pos = X - shift + 1e-8               # make all entries > 0 for NMF

    # Run NMF (temporal W, spatial H)
    nmf = NMF(
        n_components=n_components,
        init='nndsvda',
        solver='cd',
        beta_loss='frobenius',
        max_iter=1000,
        random_state=0
    )
    W = nmf.fit_transform(X_pos)   # (T × K) temporal activations
    H = nmf.components_            # (K × P) spatial weights

    # For downstream code compatibility:
    A = H.T                        # (P × K) spatial maps per component
    S = W                          # (T × K) temporal traces per component

    # Reconstruct 2D spatial maps
    spatial_maps = expand_to_full_maps(A, mask)

    #Prepare data bundle
    preproc = {"compute_dff": {"percentile": 10,"mask_used": True},
        "baseline_subtraction": {"method": "mean over time for each pixel", "applied": True},
        "bandpass_filter": {"function": "bandpass_filter_efficient", "fs": 10,"low": 0.01,"high": None,"order": 3,"batch_cols": 16000,"out_dtype": "float32"},
        "NMF": {"n_components": 100,"init":'nndsvda', "solver":'cd', "beta_loss":"frobenius","max_iter": 1000,"random_state": 0},
        "movie_path": file_data,
        "mask_path": file_mask,
        "movie_shape": movie.shape,
    }
    bundle = {
        # Analysis            
        "nmf": nmf,
        "A": A, 
        "S": S,  
        "spatial_maps": spatial_maps,
        # Movie info
        "mask": mask, 
        "mean_image": mean_img,
        "meta": {
            "frame_rate_hz": 10.0,
            "movie_shape": tuple(getattr(movie, "shape", [])),
            "source_path": file_data,
            "preprocessing": preproc,
        }
    }
    # Save the bundle
    out_path = os.path.join(path_save, f"{file_stem}_NMF_no_filt.pkl")
    with open(out_path, "wb") as f:
        pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
    return

In [4]:
data_source = {
"exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT":["pup1_spont","pup2_spont"]
}

In [ ]:
#This is the most computationally heavy step for the entire pipeline. It takes ~50min per movie
path_dist="Data"
folder_exp = "exp11_22_04_18_AL1643_P0pups_Gad2-KCC2-WT"
for file_stem in data_source[folder_exp]:
    print(file_stem)
    os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
    path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
    file_data=os.path.join(path_dist, folder_exp, file_stem, file_stem+".tif")
    file_mask=os.path.join(path_dist, folder_exp, file_stem, 'Mask.tif')

    spatial_maps=main(file_data,file_mask,path_save,file_stem, n_components=100)